<a href="https://colab.research.google.com/github/mimia3ung-lang/Blockchain/blob/main/Exercise_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import hashlib
import pickle
import time


# =========================================================
# 1. SHA-256 HASH FUNCTION
# =========================================================

def hash_data(data):
    """
    Convert data to a string, encode it as bytes,
    and generate a SHA-256 hexadecimal hash.
    """
    return hashlib.sha256(
        str(data).encode()
    ).hexdigest()


# =========================================================
# 2. MERKLE ROOT FUNCTION
# =========================================================

def create_merkle_root(transactions):
    """
    Create a simplified Merkle Root using the method
    demonstrated in the COMP842 tutorial.
    """

    if len(transactions) == 0:
        return ""

    # Hash every transaction
    hashes = [
        hash_data(transaction)
        for transaction in transactions
    ]

    # Continue until only one hash remains
    while len(hashes) > 1:

        # Duplicate the last hash if the number is odd
        if len(hashes) % 2 != 0:
            hashes.append(hashes[-1])

        next_level = []

        # Combine hashes in pairs
        for i in range(0, len(hashes), 2):
            combined = hashes[i] + hashes[i + 1]
            parent_hash = hash_data(combined)
            next_level.append(parent_hash)

        hashes = next_level

    return hashes[0]


# =========================================================
# 3. BLOCK CLASS
# =========================================================

class Block:

    def __init__(
        self,
        height,
        transactions,
        previous_hash
    ):
        self.height = height
        self.timestamp = int(time.time())
        self.previous_hash = previous_hash
        self.transactions = transactions

        # Store one Merkle Root representing all transactions
        self.merkle_root = create_merkle_root(
            self.transactions
        )

        # Generate the block hash
        self.block_hash = self.calculate_block_hash()

    def calculate_block_hash(self):
        """
        Serialize the important block fields and generate
        the SHA-256 block hash.
        """

        block_data = {
            "height": self.height,
            "timestamp": self.timestamp,
            "previous_hash": self.previous_hash,
            "merkle_root": self.merkle_root
        }

        block_bytes = pickle.dumps(block_data)

        return hashlib.sha256(
            block_bytes
        ).hexdigest()

    def display_block(self):
        print("-" * 80)
        print(f"Block Height: {self.height}")
        print(f"Timestamp: {self.timestamp}")
        print(f"Previous Hash: {self.previous_hash}")
        print(f"Transactions: {self.transactions}")
        print(f"Merkle Root: {self.merkle_root}")
        print(f"Block Hash: {self.block_hash}")


# =========================================================
# 4. BLOCKCHAIN CLASS
# =========================================================

class Blockchain:

    def __init__(self):
        self.chain = []
        self.create_genesis_block()

    def create_genesis_block(self):
        """
        Automatically create the first block.
        """

        genesis_transactions = [
            "Genesis transaction"
        ]

        genesis_block = Block(
            height=1,
            transactions=genesis_transactions,
            previous_hash="Genesis Block"
        )

        self.chain.append(genesis_block)

    def add_block(self, transactions):
        """
        Add a new block linked to the previous block.
        """

        previous_block = self.chain[-1]

        new_block = Block(
            height=len(self.chain) + 1,
            transactions=transactions,
            previous_hash=previous_block.block_hash
        )

        self.chain.append(new_block)

    def display_blockchain(self):
        for block in self.chain:
            block.display_block()

    def validate_blockchain(self):
        """
        Validate the complete blockchain and report
        the first block that fails validation.
        """

        for position, current_block in enumerate(self.chain):

            # ---------------------------------------------
            # Check 1: Recalculate the Merkle Root
            # ---------------------------------------------

            recalculated_merkle_root = create_merkle_root(
                current_block.transactions
            )

            if current_block.merkle_root != recalculated_merkle_root:
                return {
                    "valid": False,
                    "block_height": current_block.height,
                    "reason": (
                        "Transaction data does not match "
                        "the stored Merkle Root."
                    ),
                    "stored_merkle_root":
                        current_block.merkle_root,
                    "recalculated_merkle_root":
                        recalculated_merkle_root
                }

            # ---------------------------------------------
            # Check 2: Recalculate the block hash
            # ---------------------------------------------

            recalculated_block_hash = (
                current_block.calculate_block_hash()
            )

            if current_block.block_hash != recalculated_block_hash:
                return {
                    "valid": False,
                    "block_height": current_block.height,
                    "reason": (
                        "Stored block hash does not match "
                        "the recalculated block hash."
                    ),
                    "stored_block_hash":
                        current_block.block_hash,
                    "recalculated_block_hash":
                        recalculated_block_hash
                }

            # ---------------------------------------------
            # Check 3: Check the previous-hash connection
            # ---------------------------------------------

            if position > 0:
                previous_block = self.chain[position - 1]

                if (
                    current_block.previous_hash
                    != previous_block.block_hash
                ):
                    return {
                        "valid": False,
                        "block_height": current_block.height,
                        "reason": (
                            "Previous hash does not match "
                            "the hash of the previous block."
                        ),
                        "stored_previous_hash":
                            current_block.previous_hash,
                        "actual_previous_hash":
                            previous_block.block_hash
                    }

        return {
            "valid": True,
            "block_height": None,
            "reason": "The blockchain is valid."
        }


# =========================================================
# 5. CREATE A BLOCKCHAIN WITH 10 BLOCKS
# =========================================================

blockchain = Blockchain()

block_transactions = [
    [
        "Alice pays Bob $100",
        "Bob pays Charlie $25"
    ],
    [
        "Charlie pays David $50",
        "David pays Eve $10"
    ],
    [
        "Eve pays Frank $75",
        "Frank pays Grace $20"
    ],
    [
        "Grace pays Helen $40",
        "Helen pays Ian $15"
    ],
    [
        "Ian pays Jack $90",
        "Jack pays Alice $30"
    ],
    [
        "Alice pays Charlie $45",
        "Charlie pays Eve $12"
    ],
    [
        "Eve pays Grace $60",
        "Grace pays Ian $18"
    ],
    [
        "Ian pays Alice $35",
        "Alice pays David $8"
    ],
    [
        "David pays Jack $55",
        "Jack pays Bob $22"
    ]
]

# Nine additional blocks plus the genesis block = 10 blocks
for transactions in block_transactions:
    blockchain.add_block(transactions)


# =========================================================
# 6. DISPLAY AND VALIDATE THE ORIGINAL BLOCKCHAIN
# =========================================================

print("\n")
print("=" * 80)
print("ORIGINAL BLOCKCHAIN")
print("=" * 80)

blockchain.display_blockchain()

original_result = blockchain.validate_blockchain()

print("\n")
print("=" * 80)
print("ORIGINAL BLOCKCHAIN VALIDATION")
print("=" * 80)

print(f"Blockchain valid: {original_result['valid']}")
print(f"Result: {original_result['reason']}")


# =========================================================
# 7. MODIFY THE TRANSACTION DATA IN THE FIFTH BLOCK
# =========================================================

# chain[4] is the fifth block because Python starts at index 0
fifth_block = blockchain.chain[4]

print("\n")
print("=" * 80)
print("TAMPERING WITH THE FIFTH BLOCK")
print("=" * 80)

print("Original transaction:")
print(fifth_block.transactions[0])

# Modify the transaction without recalculating
# the Merkle Root or block hash
fifth_block.transactions[0] = "Grace pays Helen $4000"

print("\nModified transaction:")
print(fifth_block.transactions[0])

print("\nStored Merkle Root was not changed:")
print(fifth_block.merkle_root)

print("\nStored Block Hash was not changed:")
print(fifth_block.block_hash)


# =========================================================
# 8. DISPLAY AND VALIDATE THE MODIFIED BLOCKCHAIN
# =========================================================

print("\n")
print("=" * 80)
print("MODIFIED BLOCKCHAIN")
print("=" * 80)

blockchain.display_blockchain()

tampered_result = blockchain.validate_blockchain()

print("\n")
print("=" * 80)
print("MODIFIED BLOCKCHAIN VALIDATION")
print("=" * 80)

print(f"Blockchain valid: {tampered_result['valid']}")
print(
    f"First invalid block: "
    f"{tampered_result['block_height']}"
)
print(f"Reason: {tampered_result['reason']}")

if "stored_merkle_root" in tampered_result:
    print("\nStored Merkle Root:")
    print(tampered_result["stored_merkle_root"])

    print("\nRecalculated Merkle Root:")
    print(tampered_result["recalculated_merkle_root"])



ORIGINAL BLOCKCHAIN
--------------------------------------------------------------------------------
Block Height: 1
Timestamp: 1785797729
Previous Hash: Genesis Block
Transactions: ['Genesis transaction']
Merkle Root: 09c37d9040f38b1e0b15c7ffa9d407e947af49f7b75eed1b987221bab294932b
Block Hash: b59130d0156f7b77c3c671aefe9978f539fbb87da1b4ac211c30ceee0378d8e5
--------------------------------------------------------------------------------
Block Height: 2
Timestamp: 1785797729
Previous Hash: b59130d0156f7b77c3c671aefe9978f539fbb87da1b4ac211c30ceee0378d8e5
Transactions: ['Alice pays Bob $100', 'Bob pays Charlie $25']
Merkle Root: f0c694c05d921c7dcd84c3bd464968ad1d3ccbb2e5f6008f9473c360a679e89b
Block Hash: 90c045cb2e00363e98c269b15a304451f3d330c224fde1abd22203986640efea
--------------------------------------------------------------------------------
Block Height: 3
Timestamp: 1785797729
Previous Hash: 90c045cb2e00363e98c269b15a304451f3d330c224fde1abd22203986640efea
Transactions: ['Charli